# Chapter 28: Modern NLP with Transformers

Companion Colab notebook for **AI/ML: Zero to Hero**, Module 6, Chapter 28.

This notebook actually *runs* the HuggingFace `transformers` code that the lesson page (`nlp/ch28-modern-nlp-transformers.html`) shows as reference-only (Pyodide/WebAssembly can't load `transformers` or its PyTorch backend). Every model below is a **real pretrained checkpoint downloaded from the HuggingFace Hub and actually run** — no fabricated output.

**A note on model choice:** a few of the pipelines below (`zero-shot-classification`, `ner`) default to very large checkpoints (1.3-1.6&nbsp;GB) when no `model=` is given. To keep this notebook's download time reasonable while still using a genuine pretrained transformer for each task, we pin smaller checkpoints explicitly (`typeform/distilbert-base-uncased-mnli` for zero-shot, `dslim/bert-base-NER` for NER, `distilbert-base-uncased` for the raw tokenizer/model example). The `sentiment-analysis` pipeline uses its normal small default. Every download and every forward pass below is real.

Chapter 27 built TF-IDF vectors by hand — useful, but nowhere near what state-of-the-art NLP looks like today. Modern NLP starts from a model that already understands language, trained by someone else on a giant corpus, then adapted (or used directly) for your task. HuggingFace's `transformers` library is the standard way to do this in Python — and its `pipeline()` function gets you a production-quality model in a single line.

In [1]:
import transformers
import torch
print('transformers version:', transformers.__version__)
print('torch version:', torch.__version__)


transformers version: 4.57.6
torch version: 2.8.0+cpu


## 28.1 — Why Pretrained Models Beat Training From Scratch

Training a language model from nothing requires enormous text corpora, enormous compute, and enormous time — completely impractical for almost every real project. **Transfer learning** sidesteps this: take a model already trained on a massive general corpus (it has already learned grammar, word meaning, and world knowledge), and reuse that knowledge for your specific task — either directly, or with a small amount of additional training.

> **This is the same idea as Chapter 22's PyTorch transfer learning discussion for images** — a pretrained CNN already knows how to detect edges and shapes; a pretrained language model already knows grammar and semantics.

## 28.2 — The pipeline() API: One Line to a Working Model

`pipeline(task_name)` downloads a sensible default pretrained model for the task, wraps tokenization + inference + post-processing into one callable, and hands you back clean Python results.

In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")      # downloads a default fine-tuned model on first call
result = classifier("I love how fast this course moves!")
print(result)      # [{'label': 'POSITIVE', 'score': 0.9998...}]


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998674392700195}]


In [3]:
from transformers import pipeline

# Using a smaller MNLI checkpoint than the pipeline's ~1.6GB default (facebook/bart-large-mnli)
# to keep the download fast -- same zero-shot API and behavior, real model, real inference.
zs_classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")
result = zs_classifier(
    "This course covers tokenization, transformers, and LLM APIs.",
    candidate_labels=["education", "sports", "finance"]
)
print(result['labels'])   # ranked by confidence
print(result['scores'])   # confidence for each label, matching order above


Device set to use cpu


['education', 'finance', 'sports']
[0.5801082253456116, 0.21549507975578308, 0.20439669489860535]


> **"Zero-shot" means the model was never explicitly trained on YOUR labels** — you supply the candidate labels at inference time, and the model uses its general language understanding to rank them.

## 28.3 — AutoTokenizer & AutoModel: More Control When You Need It

`pipeline()` hides the tokenizer and model behind one call. For more control — inspecting token IDs, getting raw hidden states, building a custom head on top — use `AutoTokenizer` and `AutoModel` directly.

In [4]:
from transformers import AutoTokenizer, AutoModel

# distilbert-base-uncased instead of bert-base-uncased: same AutoTokenizer/AutoModel API,
# ~4x smaller download (~270MB vs ~440MB), same teaching point.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")

inputs = tokenizer("Hello, transformers!", return_tensors="pt")   # "pt" = return PyTorch tensors
print(inputs['input_ids'])       # token IDs the model actually sees

outputs = model(**inputs)                             # forward pass, no gradients needed for inference
print(outputs.last_hidden_state.shape)   # [1, num_tokens, hidden_size] -- one vector per token


tensor([[  101,  7592,  1010, 19081,   999,   102]])
torch.Size([1, 6, 768])


> **"Auto" means the class figures out the right architecture from the model name** — `AutoModel.from_pretrained(...)` loads whichever architecture matches the checkpoint. You rarely need to import a model-specific class by name.

## 28.4 — Named Entity Recognition: Finding People, Places, Organizations

**Named Entity Recognition (NER)** tags spans of text as belonging to a category — person, organization, location, and more.

In [5]:
from transformers import pipeline

# dslim/bert-base-NER instead of the pipeline's ~1.3GB default (bert-large-based):
# same aggregation_strategy="simple" API, ~3x smaller download.
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")   # merges sub-word pieces into whole entities
result = ner("Alice works at OpenAI in San Francisco.")

for entity in result:
    print(entity['word'], '->', entity['entity_group'])


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cpu


Alice -> PER
OpenAI -> ORG
San Francisco -> LOC


> **aggregation_strategy="simple" matters** — without it, NER models return one tag per SUB-WORD token. Aggregation merges consecutive same-entity sub-words back into whole, readable spans.

## 28.5 — Fine-Tuning: When Pretrained + Zero-Shot Isn't Enough

Sometimes a task is specialized enough that a general pretrained model, used as-is, isn't accurate enough. **Fine-tuning** continues training a pretrained model on your smaller, labeled dataset. HuggingFace's `Trainer` API is the standard tool for this — it wraps the training loop (optimizer, learning rate schedule, evaluation, checkpointing). Fine-tuning is a full topic on its own — for this course, know that the tool is called `Trainer` and what problem it solves; a runnable example is out of scope here (and out of scope for this notebook, matching the lesson).

> **Try zero-shot and pretrained pipelines FIRST** — fine-tuning needs labeled data, GPU time, and careful evaluation to avoid overfitting on a small dataset.

## Chapter 28 Summary

1. **Transfer learning** reuses a model already trained on a massive corpus instead of training from scratch.
2. **pipeline(task_name)** is the fastest way to a working pretrained model — sentiment analysis, zero-shot classification, NER, and more.
3. **AutoTokenizer / AutoModel** give direct access to tokens and hidden states when pipeline() isn't flexible enough.
4. **NER** tags spans of text as entities (person, org, location) — use `aggregation_strategy="simple"` to merge sub-words.
5. **Fine-tuning** (via the `Trainer` API) adapts a pretrained model to your labeled data when zero-shot/pretrained isn't accurate enough.

---

## Mini Project: Candidate Feedback Triage

A small but realistic use of pretrained pipelines: given short feedback comments about candidates, use sentiment analysis to flag negative ones for review, then use zero-shot classification to route each comment into a category — no training data of our own required. Both pipelines below are the same real models loaded in 28.2 above (HuggingFace caches the downloaded weights, so reusing `classifier` and `zs_classifier` here doesn't re-download anything).

In [6]:
comments = [
    "Alice communicated clearly and solved the problem fast.",
    "Bob's solution was messy and hard to follow.",
    "Carol asked great clarifying questions."
]

print("=== Sentiment ===")
sentiment_results = classifier(comments)
for comment, res in zip(comments, sentiment_results):
    print(f"{res['label']:>8}  ({res['score']:.4f})  {comment}")

print()
print("=== Category (zero-shot) ===")
candidate_labels = ["communication", "technical skill", "problem solving"]
for comment in comments:
    res = zs_classifier(comment, candidate_labels=candidate_labels)
    top_label = res['labels'][0]
    top_score = res['scores'][0]
    print(f"{top_label:>16}  ({top_score:.4f})  {comment}")

print()
positive_count = sum(1 for r in sentiment_results if r['label'] == 'POSITIVE')
negative_count = sum(1 for r in sentiment_results if r['label'] == 'NEGATIVE')
print(f"Final count: {positive_count} POSITIVE, {negative_count} NEGATIVE")


=== Sentiment ===
POSITIVE  (0.9983)  Alice communicated clearly and solved the problem fast.
NEGATIVE  (0.9998)  Bob's solution was messy and hard to follow.
POSITIVE  (0.9990)  Carol asked great clarifying questions.

=== Category (zero-shot) ===
   communication  (0.9697)  Alice communicated clearly and solved the problem fast.
   communication  (0.4320)  Bob's solution was messy and hard to follow.


   communication  (0.4136)  Carol asked great clarifying questions.

Final count: 2 POSITIVE, 1 NEGATIVE


## Next: Chapter 29 — Prompt Engineering & LLM APIs